In [22]:
import csv
import os
import re
import time

from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.options import Options

from review_parser import get_reviews


In [23]:
PRAIAS = [
    {
        "nome": "Praia Estaleiro",
        "url": "https://www.tripadvisor.com.br/Attraction_Review-g303589-d4056819-Reviews-Praia_Estaleiro-Porto_Belo_State_of_Santa_Catarina.html"
    },
        {
        "nome": "Praia Caixa D Aço",
        "url": "https://www.tripadvisor.com.br/Attraction_Review-g303589-d4056794-Reviews-Caixa_D_Aco_Beach-Porto_Belo_State_of_Santa_Catarina.html"
    },
        {
        "nome": "Praia de Porto Belo",
        "url": "https://www.tripadvisor.com.br/Attraction_Review-g303589-d4056808-Reviews-Porto_Belo_Beach-Porto_Belo_State_of_Santa_Catarina.html"
    },
        {
        "nome": "Praia Araçá",
        "url": "https://www.tripadvisor.com.br/Attraction_Review-g303589-d4056809-Reviews-Araca_Beach-Porto_Belo_State_of_Santa_Catarina.html"
    },
        {
        "nome": "Praia Perequê",
        "url": "https://www.tripadvisor.com.br/Attraction_Review-g303589-d4056805-Reviews-Pereque_Beach-Porto_Belo_State_of_Santa_Catarina.html"
    }
]


In [24]:
BASE_URL = "https://www.tripadvisor.com.br"


def get_driver_connection():
    options = Options()
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_argument("--window-size=1920,1080")
    options.add_argument("--disable-extensions")
    options.add_argument("--disable-popup-blocking")
    options.add_argument("--disable-plugins-discovery")
    options.add_experimental_option("excludeSwitches", ["enable-automation"])
    options.add_experimental_option("useAutomationExtension", False)
    options.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    )

    driver = webdriver.Chrome(options=options)
    driver.execute_cdp_cmd(
        "Page.addScriptToEvaluateOnNewDocument",
        {
            "source": """
                Object.defineProperty(navigator, 'webdriver', {get: () => undefined});
                Object.defineProperty(navigator, 'plugins', {get: () => [1, 2, 3]});
                Object.defineProperty(navigator, 'languages', {get: () => ['pt-BR', 'pt', 'en-US']});
                window.chrome = { runtime: {} };
            """
        },
    )
    return driver


def get_next_url(soup: BeautifulSoup) -> str | None:
    proxima = soup.find("a", {"data-smoke-attr": "pagination-next-arrow"})
    if proxima and proxima.get("href"):
        href = proxima["href"]

        if href.startswith("http"):
            return href

        return BASE_URL + href

    return None


def sanitize_filename(texto: str) -> str:
    texto = texto.lower().strip()
    texto = re.sub(r"[^\w\s-]", "", texto)
    texto = re.sub(r"[\s-]+", "_", texto)
    return texto


def save_reviews_to_csv(nome_praia: str, reviews: list[dict]):
    os.makedirs("csv", exist_ok=True)

    nome_arquivo = sanitize_filename(nome_praia)
    caminho = os.path.join("csv", f"{nome_arquivo}.csv")

    with open(caminho, "w", newline="", encoding="utf-8-sig") as arquivo:
        writer = csv.DictWriter(
            arquivo,
            fieldnames=[
                "rating",
                "title",
                "text",
                "location",
                "num_contributions",
                "date",
            ],
        )
        writer.writeheader()
        writer.writerows(reviews)

    print(f"CSV criado: {caminho}")


In [25]:
def scrape_praia(driver, nome_praia: str, primeira_url: str) -> list[dict]:
    todas_reviews = []

    # Define a primeira URL que será acessada
    url_atual = primeira_url

    pagina = 1

    # Guarda chaves para evitar reviews duplicadas
    vistos = set()

    # Continua enquanto houver próxima página
    while url_atual:
        print(f"\nPraia: {nome_praia} | Página {pagina}")
        print(url_atual)

        # Abre a página no navegador
        driver.get(url_atual)

        time.sleep(4)

        driver.execute_script("window.scrollTo(0, 300);")
        time.sleep(1.5)
        driver.execute_script("window.scrollTo(0, 700);")
        time.sleep(1.5)

        # Converte o HTML carregado em objeto BeautifulSoup
        soup = BeautifulSoup(driver.page_source, "html.parser")

        # Extrai as reviews da página
        reviews = get_reviews(soup)

        reviews_filtradas = []

        # Percorre cada review
        for review in reviews:
            chave = (
                review.get("title", ""),
                review.get("text", ""),
                review.get("date", ""),
            )

            if chave not in vistos:
                vistos.add(chave)
                reviews_filtradas.append(review)

        todas_reviews.extend(reviews_filtradas)

        print(f"Reviews extraídas nesta página: {len(reviews)}")
        print(f"Total acumulado: {len(todas_reviews)}")

        # Busca a URL da próxima página
        proxima_url = get_next_url(soup)

        if not proxima_url or proxima_url == url_atual:
            print("Última página atingida.")
            break

        # Atualiza para a próxima página
        url_atual = proxima_url
        pagina += 1

    return todas_reviews


In [26]:
driver = get_driver_connection()

try:
    driver.get("https://www.google.com.br")
    time.sleep(2)

    for praia in PRAIAS:
        nome_praia = praia["nome"]
        url = praia["url"]

        print("\n" + "=" * 70)
        print(f"Processando praia: {nome_praia}")
        print("=" * 70)

        reviews = scrape_praia(driver, nome_praia, url)
        print(reviews[0])
        save_reviews_to_csv(nome_praia, reviews)

finally:
    driver.quit()
    print("\nNavegador fechado.")



Processando praia: Praia Estaleiro

Praia: Praia Estaleiro | Página 1
https://www.tripadvisor.com.br/Attraction_Review-g303589-d4056819-Reviews-Praia_Estaleiro-Porto_Belo_State_of_Santa_Catarina.html
Reviews extraídas nesta página: 10
Total acumulado: 10

Praia: Praia Estaleiro | Página 2
https://www.tripadvisor.com.br/Attraction_Review-g303589-d4056819-Reviews-or10-Praia_Estaleiro-Porto_Belo_State_of_Santa_Catarina.html
Reviews extraídas nesta página: 10
Total acumulado: 20

Praia: Praia Estaleiro | Página 3
https://www.tripadvisor.com.br/Attraction_Review-g303589-d4056819-Reviews-or20-Praia_Estaleiro-Porto_Belo_State_of_Santa_Catarina.html
Reviews extraídas nesta página: 10
Total acumulado: 30

Praia: Praia Estaleiro | Página 4
https://www.tripadvisor.com.br/Attraction_Review-g303589-d4056819-Reviews-or30-Praia_Estaleiro-Porto_Belo_State_of_Santa_Catarina.html
Reviews extraídas nesta página: 10
Total acumulado: 40

Praia: Praia Estaleiro | Página 5
https://www.tripadvisor.com.br/Att